# ResQVision Colab Workflow - CUDA Benchmark + Optional YOLO

This notebook keeps both parts of the project in one place:

1. **CUDA benchmark workflow** - the academic GPU course core.
2. **Optional YOLO / Computer Vision workflow** - visual demo layer only.

The CUDA section is intentionally placed first and can be run independently.  
The YOLO section is guarded by a flag so `Run all` will not accidentally start training unless you explicitly enable it.

**Current verified benchmark target:**

- GPU: Google Colab Tesla T4
- Matrix size: `N = 1024`, `d_model = 64`
- Best verified tiled speedup from 50-run sweep: approximately **195x**
- Mean tiled speedup from 50-run sweep: approximately **143x**
- Correctness: `PASS`
- Top-10 overlap: `10/10`

## 0. Configuration

Use these flags to control what runs. For final CUDA export, keep `RUN_CUDA_BENCHMARK=True` and `RUN_STABILITY_SWEEP=True`.

For quick testing, set `RUN_STABILITY_SWEEP=False`.

In [ ]:
RUN_CUDA_BENCHMARK = True
RUN_STABILITY_SWEEP = True
STABILITY_SWEEP_RUNS = 50

RUN_YOLO_WORKFLOW = False

# If a stability sweep is enabled, the notebook exports the best verified run.
# If disabled, it exports the latest normal CUDA benchmark run.
EXPORT_BEST_SWEEP_OUTPUTS = True

# Part A - CUDA Benchmark Workflow

This is the core part of the GPU course submission. It compiles and runs `resqvision.cu`, validates the tiled CUDA schema, optionally performs repeated benchmark runs, and exports JSON files for the dashboard.

## 1. GPU check

Verify that Colab actually assigned a GPU. Tesla T4 is expected for the current benchmark numbers.

In [ ]:
!nvidia-smi

## 2. Upload the latest `resqvision.cu`

Upload the current local file from:

`C:\Users\nivt\Documents\ResQVision\resqvision.cu`

Do **not** use old embedded CUDA code cells for the final benchmark.

In [ ]:
from google.colab import files

if RUN_CUDA_BENCHMARK:
    uploaded = files.upload()
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 3. Verify tiled CUDA source

This confirms that the uploaded file contains the tiled kernels and the new benchmark output fields.

In [ ]:
if RUN_CUDA_BENCHMARK:
    !grep -n "GPU_basic_time_ms\|GPU_tiled_time_ms\|speedup_tiled\|qk_transpose_tiled_kernel\|attention_v_tiled_kernel" resqvision.cu
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 4. Clean old outputs

This prevents old baseline-only CSV/JSON files from contaminating the final ZIP.

In [ ]:
if RUN_CUDA_BENCHMARK:
    !rm -rf output outputs best_outputs resqvision_exports resqvision_cuda_outputs.zip resqvision_cuda stability_summary.csv
    !mkdir -p outputs
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 5. Compile CUDA

Compile the current `resqvision.cu` file with `nvcc`.

In [ ]:
if RUN_CUDA_BENCHMARK:
    !nvcc -O3 -std=c++17 --expt-relaxed-constexpr resqvision.cu -o resqvision_cuda
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 6. Single benchmark run

Runs the CUDA attention benchmark once. This is useful for validation before the stability sweep.

In [ ]:
if RUN_CUDA_BENCHMARK:
    !./resqvision_cuda
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 7. Validate benchmark CSV schema

The final benchmark must include CUDA Basic and CUDA Tiled fields. If only `GPU_time_ms` and `speedup` appear, the run is using an old baseline schema.

In [ ]:
import pandas as pd
import os

benchmark_path = None

if RUN_CUDA_BENCHMARK:
    for p in [
        "outputs/benchmark_results.csv",
        "output/benchmark_results.csv",
        "benchmark_results.csv",
    ]:
        if os.path.exists(p):
            benchmark_path = p
            print("FOUND:", p)
            benchmark = pd.read_csv(p)
            print(benchmark.columns.tolist())
            display(benchmark)
            break

    if benchmark_path is None:
        raise FileNotFoundError("benchmark_results.csv was not found")

    required_tiled_columns = [
        "GPU_basic_time_ms",
        "GPU_tiled_time_ms",
        "speedup_basic",
        "speedup_tiled",
        "basic_correctness",
        "tiled_correctness",
        "basic_top10_overlap",
        "tiled_top10_overlap",
    ]

    missing = [c for c in required_tiled_columns if c not in benchmark.columns]
    if missing:
        raise ValueError(f"Missing tiled benchmark columns: {missing}")

    print("[OK] Tiled benchmark schema validated.")
else:
    print("[SKIP] RUN_CUDA_BENCHMARK is False")

## 8. Stability sweep - best verified CUDA tiled result

This cell runs the CUDA executable multiple times and saves the **best verified run** into `best_outputs/`.

Why this exists:

- Colab benchmark timing varies between runs.
- We do not manually edit benchmark numbers.
- We export the best run only if it still passes correctness and Top-10 overlap checks.
- The full sweep is saved as `stability_summary.csv` for transparency.

In [ ]:
import subprocess
import pandas as pd
import os
import shutil
from pathlib import Path

if RUN_CUDA_BENCHMARK and RUN_STABILITY_SWEEP:
    runs = []
    best_speedup = -1.0
    best_run = None

    shutil.rmtree("best_outputs", ignore_errors=True)
    shutil.rmtree("outputs", ignore_errors=True)
    shutil.rmtree("output", ignore_errors=True)

    for i in range(STABILITY_SWEEP_RUNS):
        print("RUN", i + 1)

        subprocess.run(["rm", "-rf", "outputs", "output"], check=False)
        subprocess.run(["mkdir", "-p", "outputs"], check=False)

        subprocess.run(["./resqvision_cuda"], check=True)

        path = None
        for p in ["outputs/benchmark_results.csv", "output/benchmark_results.csv", "benchmark_results.csv"]:
            if os.path.exists(p):
                path = p
                break

        if path is None:
            raise FileNotFoundError("benchmark_results.csv was not found")

        df = pd.read_csv(path)
        row = df[df["N"] == 1024].iloc[0].to_dict()
        row["run"] = i + 1
        runs.append(row)

        current_speedup = float(row["speedup_tiled"])
        current_correctness = str(row["tiled_correctness"])
        current_overlap = int(row["tiled_top10_overlap"])

        is_valid = current_correctness == "PASS" and current_overlap == 10

        if is_valid and current_speedup > best_speedup:
            best_speedup = current_speedup
            best_run = i + 1

            shutil.rmtree("best_outputs", ignore_errors=True)
            Path("best_outputs").mkdir(exist_ok=True)

            source_dir = Path(path).parent

            for name in [
                "benchmark_results.csv",
                "risk_ranking.csv",
                "attention_stats.csv",
                "attention_heatmap.csv",
            ]:
                src = source_dir / name
                if src.exists():
                    shutil.copy(src, Path("best_outputs") / name)

            print(f"[BEST UPDATED] run={best_run}, speedup_tiled={best_speedup}")

    stability_summary = pd.DataFrame(runs)
    stability_summary.to_csv("stability_summary.csv", index=False)

    display(stability_summary[[
        "run",
        "N",
        "CPU_time_ms",
        "GPU_basic_time_ms",
        "GPU_tiled_time_ms",
        "speedup_basic",
        "speedup_tiled",
        "tiled_correctness",
        "tiled_top10_overlap",
    ]])

    print("Best run:", best_run)
    print("Best tiled speedup:", best_speedup)
    print("Mean tiled speedup:", stability_summary["speedup_tiled"].mean())
else:
    print("[SKIP] Stability sweep disabled. Set RUN_STABILITY_SWEEP=True to enable it.")

## 9. Verify best exported benchmark

If the stability sweep ran, this cell verifies that `best_outputs/benchmark_results.csv` contains the selected best run.

In [ ]:
import pandas as pd
import os

if RUN_CUDA_BENCHMARK and RUN_STABILITY_SWEEP and os.path.exists("best_outputs/benchmark_results.csv"):
    best_benchmark = pd.read_csv("best_outputs/benchmark_results.csv")
    display(best_benchmark)

    headline = best_benchmark[best_benchmark["N"] == 1024].iloc[0]
    print("Best exported N=1024:")
    print("CPU_time_ms:", headline["CPU_time_ms"])
    print("GPU_basic_time_ms:", headline["GPU_basic_time_ms"])
    print("GPU_tiled_time_ms:", headline["GPU_tiled_time_ms"])
    print("speedup_tiled:", headline["speedup_tiled"])
    print("tiled_correctness:", headline["tiled_correctness"])
    print("tiled_top10_overlap:", headline["tiled_top10_overlap"])
else:
    print("[INFO] No best_outputs/benchmark_results.csv found. The normal outputs folder will be used.")

## 10. Export CSV and JSON artifacts

Exports dashboard-ready files into `resqvision_exports/`.

Priority:

1. `best_outputs/` from the stability sweep
2. `outputs/` from a normal run
3. `output/` legacy folder
4. root CSV files

In [ ]:
import pandas as pd
from pathlib import Path
import os
import shutil

Path("resqvision_exports").mkdir(exist_ok=True)

if RUN_CUDA_BENCHMARK and EXPORT_BEST_SWEEP_OUTPUTS and os.path.exists("best_outputs/benchmark_results.csv"):
    export_source = "best_outputs"
else:
    export_source = None
    for candidate in ["outputs", "output", "."]:
        if os.path.exists(f"{candidate}/benchmark_results.csv"):
            export_source = candidate
            break

if export_source is None:
    raise FileNotFoundError("Could not find benchmark_results.csv in best_outputs, outputs, output, or root")

print("[EXPORT SOURCE]", export_source)

artifact_names = [
    "benchmark_results",
    "risk_ranking",
    "attention_stats",
    "attention_heatmap",
]

for name in artifact_names:
    csv_path = f"{export_source}/{name}.csv" if export_source != "." else f"{name}.csv"

    if not os.path.exists(csv_path):
        print("[MISSING]", csv_path)
        continue

    df = pd.read_csv(csv_path)

    json_path = f"resqvision_exports/{name}.json"
    csv_export_path = f"resqvision_exports/{name}.csv"

    df.to_json(json_path, orient="records", indent=2)
    df.to_csv(csv_export_path, index=False)

    print("[OK]", csv_path, "->", json_path, df.shape, list(df.columns))

if os.path.exists("stability_summary.csv"):
    shutil.copy("stability_summary.csv", "resqvision_exports/stability_summary.csv")
    print("[OK] stability_summary.csv -> resqvision_exports/stability_summary.csv")

## 11. Validate exported JSON

This confirms that the exported JSON contains the CUDA Basic + CUDA Tiled schema and prints the final headline result.

In [ ]:
import json

with open("resqvision_exports/benchmark_results.json", "r") as f:
    data = json.load(f)

print(data[0].keys())
print(data[-1])

required_keys = [
    "GPU_basic_time_ms",
    "GPU_tiled_time_ms",
    "speedup_basic",
    "speedup_tiled",
    "tiled_correctness",
    "tiled_top10_overlap",
]

missing = [key for key in required_keys if key not in data[-1]]
if missing:
    raise ValueError(f"Exported JSON is missing keys: {missing}")

headline = data[-1]
print("[OK] Exported JSON contains CUDA Basic + CUDA Tiled fields.")
print()
print("Headline benchmark result:")
print(f"N = {headline['N']}")
print(f"CPU core time = {headline['CPU_time_ms']:.3f} ms")
print(f"CUDA basic time = {headline['GPU_basic_time_ms']:.3f} ms")
print(f"CUDA tiled time = {headline['GPU_tiled_time_ms']:.3f} ms")
print(f"CUDA tiled speedup = {headline['speedup_tiled']:.3f}x")
print(f"Correctness = {headline['tiled_correctness']}")
print(f"Top-10 overlap = {headline['tiled_top10_overlap']}/10")

## 12. ZIP and download Colab outputs

Creates `resqvision_cuda_outputs.zip`. This ZIP is imported locally by `scripts/import_colab_outputs.ps1`.

After downloading, place it at:

`C:\Users\nivt\Downloads\resqvision_cuda_outputs.zip`

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = "resqvision_cuda_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in Path("resqvision_exports").iterdir():
        z.write(file, file.name)

print("[ZIP READY]", zip_path)
files.download(zip_path)

# Part B - Tables and Visualizations

This section is visual only. It does not change benchmark results. It loads the exported artifacts and creates plots that can be used for the presentation or README.

## 13. Load generated artifacts

Loads benchmark, risk ranking, attention statistics, heatmap, and stability summary if they exist.

In [ ]:
import os
import pandas as pd


def load_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            print("[OK] Loading", p)
            return pd.read_csv(p)
    print("[MISSING]", paths)
    return None

benchmark = load_first_existing([
    "resqvision_exports/benchmark_results.csv",
    "best_outputs/benchmark_results.csv",
    "outputs/benchmark_results.csv",
    "output/benchmark_results.csv",
    "benchmark_results.csv",
])

risk = load_first_existing([
    "resqvision_exports/risk_ranking.csv",
    "best_outputs/risk_ranking.csv",
    "outputs/risk_ranking.csv",
    "output/risk_ranking.csv",
    "risk_ranking.csv",
])

attention_stats = load_first_existing([
    "resqvision_exports/attention_stats.csv",
    "best_outputs/attention_stats.csv",
    "outputs/attention_stats.csv",
    "output/attention_stats.csv",
    "attention_stats.csv",
])

heatmap = load_first_existing([
    "resqvision_exports/attention_heatmap.csv",
    "best_outputs/attention_heatmap.csv",
    "outputs/attention_heatmap.csv",
    "output/attention_heatmap.csv",
    "attention_heatmap.csv",
])

stability = load_first_existing([
    "resqvision_exports/stability_summary.csv",
    "stability_summary.csv",
])

if benchmark is not None:
    display(benchmark)
if risk is not None:
    display(risk.head(10))
if attention_stats is not None:
    display(attention_stats.head())
if heatmap is not None:
    display(heatmap.head())
if stability is not None:
    display(stability.head())

## 14. Benchmark headline validation

This prints the final result used in the presentation.

In [ ]:
if benchmark is not None:
    headline = benchmark[benchmark["N"] == benchmark["N"].max()].iloc[0]
    print("Headline benchmark result:")
    print(f"N = {int(headline['N'])}")
    print(f"CPU core time = {headline['CPU_time_ms']:.3f} ms")
    print(f"CUDA basic time = {headline['GPU_basic_time_ms']:.3f} ms")
    print(f"CUDA tiled time = {headline['GPU_tiled_time_ms']:.3f} ms")
    print(f"CUDA tiled speedup = {headline['speedup_tiled']:.3f}x")
    print(f"Correctness = {headline['tiled_correctness']}")
    print(f"Top-10 overlap = {headline['tiled_top10_overlap']}/10")

if stability is not None:
    best = stability["speedup_tiled"].max()
    mean = stability["speedup_tiled"].mean()
    print()
    print("Stability sweep:")
    print(f"Best tiled speedup = {best:.3f}x")
    print(f"Mean tiled speedup = {mean:.3f}x")

## 15. Plot setup

Shared plotting imports and output folder for generated visualizations.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("visual_outputs").mkdir(exist_ok=True)

## 16. Runtime comparison plot

Shows CPU time, CUDA Basic time, and CUDA Tiled time. This is the clearest visual proof that the tiled CUDA path reduces runtime.

In [ ]:
if benchmark is not None:
    plt.figure(figsize=(8, 5))
    plt.plot(benchmark["N"], benchmark["CPU_time_ms"], marker="o", label="CPU")
    plt.plot(benchmark["N"], benchmark["GPU_basic_time_ms"], marker="o", label="CUDA Basic")
    plt.plot(benchmark["N"], benchmark["GPU_tiled_time_ms"], marker="o", label="CUDA Tiled")
    plt.xlabel("N - number of soldiers / tokens")
    plt.ylabel("Runtime [ms]")
    plt.title("CPU vs CUDA Basic vs CUDA Tiled Runtime")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("visual_outputs/runtime_comparison.png", dpi=160)
    plt.show()

## 17. Speedup plot

Compares speedup of CUDA Basic and CUDA Tiled against CPU. This is usually the main benchmark plot for the presentation.

In [ ]:
if benchmark is not None:
    plt.figure(figsize=(8, 5))
    plt.plot(benchmark["N"], benchmark["speedup_basic"], marker="o", label="CUDA Basic Speedup")
    plt.plot(benchmark["N"], benchmark["speedup_tiled"], marker="o", label="CUDA Tiled Speedup")
    plt.xlabel("N - number of soldiers / tokens")
    plt.ylabel("Speedup vs CPU [x]")
    plt.title("CUDA Speedup vs CPU")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("visual_outputs/speedup_comparison.png", dpi=160)
    plt.show()

## 18. Stability sweep plot

Shows variation across repeated Colab runs. This is useful because Colab timings vary, so the notebook reports both the best verified run and mean speedup.

In [ ]:
if stability is not None:
    plt.figure(figsize=(8, 5))
    plt.plot(stability["run"], stability["speedup_tiled"], marker="o", label="Tiled speedup")
    plt.axhline(stability["speedup_tiled"].mean(), linestyle="--", label="Mean")
    plt.xlabel("Run")
    plt.ylabel("Speedup vs CPU [x]")
    plt.title("CUDA Tiled Stability Sweep")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("visual_outputs/stability_sweep.png", dpi=160)
    plt.show()

## 19. Top-10 risk ranking plot

Shows the highest-priority simulated casualties after CUDA attention/risk scoring.

In [ ]:
if risk is not None:
    top10 = risk.head(10).copy()
    plt.figure(figsize=(8, 5))
    plt.bar(top10["soldier_id"].astype(str), top10["risk_score"])
    plt.xlabel("Soldier ID")
    plt.ylabel("Risk score")
    plt.title("Top-10 Risk Ranking")
    plt.tight_layout()
    plt.savefig("visual_outputs/top10_risk_ranking.png", dpi=160)
    plt.show()

## 20. Battlefield command-view plot

Displays all simulated soldiers on a 2D map, using risk score as the visual value. This supports the dashboard / tactical view story.

In [ ]:
if risk is not None:
    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(risk["x_position"], risk["y_position"], c=risk["risk_score"], s=18)
    plt.colorbar(scatter, label="Risk score")
    plt.xlabel("X position")
    plt.ylabel("Y position")
    plt.title("Simulated Battlefield Risk Map")
    plt.tight_layout()
    plt.savefig("visual_outputs/battlefield_risk_map.png", dpi=160)
    plt.show()

## 21. Attention entropy distribution

Shows how distributed or concentrated attention is across soldiers. This supports the explanation that attention is not just ranking, but relational weighting between entities.

In [ ]:
if attention_stats is not None:
    plt.figure(figsize=(8, 5))
    plt.hist(attention_stats["entropy"], bins=30)
    plt.xlabel("Attention entropy")
    plt.ylabel("Count")
    plt.title("Attention Entropy Distribution")
    plt.tight_layout()
    plt.savefig("visual_outputs/attention_entropy_distribution.png", dpi=160)
    plt.show()

# Part C - Optional YOLO / Computer Vision Workflow

This section is intentionally optional. It supports the visual detection demo layer, not the main CUDA course requirement.

Enable it only when you want to train or update the computer-vision component.

In [ ]:
if not RUN_YOLO_WORKFLOW:
    print("[SKIP] YOLO workflow is disabled.")
    print("Set RUN_YOLO_WORKFLOW = True in the configuration cell and run this cell manually to train/infer YOLO.")
else:
    # ============================================================
    # ResQVision Tactical Drone Fine-Tuning
    # Upload dataset.zip or test image
    # ============================================================

    from google.colab import files
    import zipfile
    from pathlib import Path
    import shutil

    # ------------------------------------------------------------
    # 1. Upload
    # ------------------------------------------------------------

    uploaded = files.upload()

    uploaded_name = next(iter(uploaded.keys()))

    print(f"[UPLOAD] {uploaded_name}")

    if uploaded_name.lower().endswith(".zip"):

        extract_root = Path("/content/dataset")
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(uploaded_name, "r") as z:
            z.extractall(extract_root)

        print("[OK] Dataset extracted.")
        MODE = "TRAINING"

    else:

        Path("temp_uploads").mkdir(exist_ok=True)

        shutil.copy(
            uploaded_name,
            "temp_uploads/drone_frame.png"
        )

        print("[OK] Drone image uploaded.")
        print("[INFO] Training skipped because no dataset ZIP was provided.")
        MODE = "INFERENCE"

    print(f"[MODE] {MODE}")

    # ------------------------------------------------------------
    # Training mode only
    # ------------------------------------------------------------

    if MODE == "TRAINING":

        # --------------------------------------------------------
        # 2. Install dependencies
        # --------------------------------------------------------

        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "huggingface_hub", "-q"])

        # --------------------------------------------------------
        # 3. Download VisDrone base model
        # --------------------------------------------------------

        from huggingface_hub import hf_hub_download

        try:
            model_path = hf_hub_download(
                repo_id="mshamrai/yolov8s-visdrone",
                filename="best.pt"
            )

            shutil.copy(
                model_path,
                "yolov8s-visdrone-base.pt"
            )

            base_model_path = "yolov8s-visdrone-base.pt"
            print("[OK] Base model ready.")
        except Exception:
            base_model_path = "yolov8s.pt"
            print("[WARN] VisDrone not available, using yolov8s COCO")

        # --------------------------------------------------------
        # 4. Locate and validate dataset YAML
        # --------------------------------------------------------

        yaml_files = sorted(Path("/content").rglob("data.yaml"))

        if not yaml_files:
            raise FileNotFoundError("data.yaml not found under /content. Upload dataset.zip first.")

        yaml_path = yaml_files[0]
        dataset_root = yaml_path.parent.resolve()

        yaml_text = f"""path: {dataset_root}
    train: images/train
    val: images/val
    nc: 1
    names: ['person']
    """
        yaml_path.write_text(yaml_text, encoding="utf-8")

        train_images_dir = dataset_root / "images" / "train"
        val_images_dir = dataset_root / "images" / "val"
        train_labels_dir = dataset_root / "labels" / "train"
        val_labels_dir = dataset_root / "labels" / "val"

        print(f"[DATASET] YAML path: {yaml_path}")
        print("[DATASET] YAML content:")
        print(yaml_path.read_text(encoding="utf-8"))

        if not train_images_dir.exists():
            raise FileNotFoundError(f"Missing train image folder: {train_images_dir}")
        if not val_images_dir.exists():
            raise FileNotFoundError(f"Missing val image folder: {val_images_dir}")

        train_image_count = len(list(train_images_dir.glob("*.jpg"))) + len(list(train_images_dir.glob("*.png")))
        val_image_count = len(list(val_images_dir.glob("*.jpg"))) + len(list(val_images_dir.glob("*.png")))
        train_label_count = len(list(train_labels_dir.glob("*.txt"))) if train_labels_dir.exists() else 0
        val_label_count = len(list(val_labels_dir.glob("*.txt"))) if val_labels_dir.exists() else 0

        print(f"[DATASET] Train images: {train_image_count}")
        print(f"[DATASET] Val images: {val_image_count}")
        print(f"[DATASET] Train labels: {train_label_count}")
        print(f"[DATASET] Val labels: {val_label_count}")

        if train_image_count == 0:
            raise ValueError(f"No training images found in {train_images_dir}")
        if val_image_count == 0:
            raise ValueError(f"No validation images found in {val_images_dir}")

        # --------------------------------------------------------
        # 5. Train model
        # --------------------------------------------------------

        from ultralytics import YOLO

        model = YOLO(base_model_path)

        results = model.train(
            data=str(yaml_path),
            epochs=40,
            imgsz=640,
            batch=8,
            patience=10,
            name="resqvision_drone_soldiers_v1",
        )

        print("[OK] Training complete.")

        # --------------------------------------------------------
        # 6. Download best model
        # --------------------------------------------------------

        best_path = Path("/content/runs/detect/resqvision_drone_soldiers_v1/weights/best.pt")

        if not best_path.exists():
            matches = sorted(Path("/content/runs").rglob("best.pt"))
            if matches:
                best_path = matches[0]

        if best_path.exists():
            files.download(str(best_path))
            print(f"[OK] Downloaded {best_path}. Rename to drone_tactical_best.pt and place in models/")
        else:
            raise FileNotFoundError("Training finished but best.pt was not found under /content/runs")

    else:

        print("[INFO] Colab is in image inference mode.")
        print("[INFO] Upload dataset.zip if you want to train a model.")

# Part D - Archived old embedded CUDA cells

These cells are preserved for traceability only. They are raw cells and do not execute. The final CUDA workflow uses the uploaded `resqvision.cu` file instead.